# Breast Cancer scRNA-seq: Data Processing Pipeline

Full preprocessing pipeline for GSE161529 — downloading, loading, merging, QC filtering, normalization, dimensionality reduction, clustering, and cell type annotation for 13 normal and 8 TNBC tumor samples.

**Note:** Raw data must be downloaded from GEO before running this notebook.
Dataset: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE161529   
Total runtime: ~45-60 minutes, depending on connection speed.

## Setup

In [2]:
# import required libraries
import scanpy as sc
import os

# create data directories if they don't exist
os.makedirs('../raw_data', exist_ok=True)
os.makedirs('../data', exist_ok=True)

print("Directories ready")
print("Scanpy version:", sc.__version__)

Directories ready
Scanpy version: 1.11.5


/var/folders/m0/my66jx5974n5jwyr68bndpg80000gn/T/ipykernel_82465/2035273673.py:9: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print("Scanpy version:", sc.__version__)


In [ ]:
## Data Download

Raw data downloaded from GEO (GSE161529). This step is skipped if the file already exists locally. The full dataset is ~2.36 GB.

In [ ]:
import urllib.request
import os

url = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE161nnn/GSE161529/suppl/GSE161529_RAW.tar"
output_path = "../raw_data/GSE161529_RAW.tar"

# skip download if file already exists
if os.path.exists(output_path):
    print(f"File already exists ({os.path.getsize(output_path) / 1e9:.2f} GB), skipping download")
else:
    print("Downloading dataset...")
    urllib.request.urlretrieve(url, output_path)
    print(f"Download complete, File size: {os.path.getsize(output_path) / 1e9:.2f} GB")

In [ ]:
import tarfile
import os

# skip extraction if files already exist to avoid re-extracting on re-run
if len([f for f in os.listdir("../raw_data/") if f.endswith('.mtx.gz')]) > 0:
    print("Files already extracted, skipping")
    files = os.listdir("../raw_data/")
else:
    print("Extracting files...")
    with tarfile.open("../raw_data/GSE161529_RAW.tar", "r") as tar:
        tar.extractall("../raw_data/")
    files = os.listdir("../raw_data/")
    print(f"Extraction complete! Found {len(files)} files")

## Sample Identification
The dataset contains multiple breast cancer subtypes. We focus on Normal (N) vs Triple Negative (TN) samples for this analysis. Tumor samples use the prefix '_TN-'.

In [ ]:
# identify normal and TNBC tumor samples by filename pattern
# normal samples contain '_N-' and 'Total', tumor samples contain '_TN-'
normal = [f for f in files if f.startswith('GSM') and '_N-' in f and 'Total' in f and 'matrix' in f]
tumor_tn = [f for f in files if '_TN-' in f and 'matrix' in f]

print(f"Normal samples: {len(normal)}")
print(f"Triple Negative tumor samples: {len(tumor_tn)}")

## Data Loading

Each sample consists of a sparse count matrix (.mtx.gz) and a cell barcode file (.barcodes.tsv.gz). We define a function to load each sample and attach condition labels (Normal or Tumor_TN) before merging all 21 samples.

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import os

raw_data_path = "../raw_data/"

def load_sample(sample_name, condition):
    """Load 10x Genomics sample from barcodes + matrix files"""
    # strip the matrix filename suffix to get the base sample name
    base = sample_name.replace('-matrix.mtx.gz', '')
    
    matrix_file = os.path.join(raw_data_path, f"{base}-matrix.mtx.gz")
    barcodes_file = os.path.join(raw_data_path, f"{base}-barcodes.tsv.gz")
    
    # read sparse count matrix and transpose so cells are rows, genes are columns
    adata = sc.read_mtx(matrix_file).T
    
    # read cell barcodes and assign as observation names
    barcodes = pd.read_csv(barcodes_file, header=None)[0].values
    adata.obs_names = barcodes
    
    # attach sample ID and condition label to each cell
    adata.obs['sample'] = base
    adata.obs['condition'] = condition
    
    return adata

print("Ready to load samples")

In [ ]:
# load each normal sample using the load_sample function and store in a list
print("Loading normal samples...")
normal_adatas = []
for f in sorted(normal):
    print(f"  Loading {f}...")
    adata = load_sample(f, 'Normal')
    normal_adatas.append(adata)
print(f"\nLoaded {len(normal_adatas)} normal samples")

# load each TNBC tumor sample
print("\nLoading tumor samples...")
tumor_adatas = []
for f in sorted(tumor_tn):
    print(f"  Loading {f}...")
    adata = load_sample(f, 'Tumor_TN')
    tumor_adatas.append(adata)
print(f"\nLoaded {len(tumor_adatas)} tumor samples")

In [ ]:
# combine all 21 samples into one AnnData object
# batch_key='sample_id' creates a column tracking which sample each cell came from
print("Merging all samples...")
all_adatas = normal_adatas + tumor_adatas
adata = all_adatas[0].concatenate(all_adatas[1:], batch_key='sample_id')

# confirm merge was successful and check condition balance
print(adata)
print(f"\nCondition counts:")
print(adata.obs['condition'].value_counts())

## Gene Name Assignment

The raw files were missing the features.tsv.gz file that normally contains gene names — a common issue with older GEO submissions. We download it separately and attach the gene names to the AnnData object.

In [ ]:
# check gene names - at this point they are just numbers (0, 1, 2...)
# because the raw files were missing the features/gene names file
print("Gene names:", adata.var_names[:10].tolist())
print("Total genes:", adata.n_vars)

In [ ]:
import urllib.request

features_url = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE161nnn/GSE161529/suppl/GSE161529_features.tsv.gz"
features_path = "../raw_data/GSE161529_features.tsv.gz"

# skip download if file already exists
if os.path.exists(features_path):
    print("Features file already exists, skipping download")
else:
    print("Downloading features file...")
    urllib.request.urlretrieve(features_url, features_path)
    print("Done!")

In [ ]:
import pandas as pd

# load features file — contains 3 columns: Ensembl ID, gene name, and feature type
features = pd.read_csv("../raw_data/GSE161529_features.tsv.gz", 
                       header=None, sep='\t')
print("Features file shape:", features.shape)
print(features.head())

In [ ]:
# column 1 contains human-readable gene names (e.g. KRT18, CD74)
# column 0 contains Ensembl IDs (e.g. ENSG00000...) stored as gene_ids for reference
adata.var_names = features[1].values
adata.var['gene_ids'] = features[0].values

print("Gene names added!")
print("First 5 genes:", adata.var_names[:5].tolist())
print(adata)

In [ ]:
# save raw merged object before QC filtering
adata.write('../data/breast_cancer_raw.h5ad')
print("Saved!")

## Quality Control

Filtering out low quality cells and lowly expressed genes before normalization. 
Cells with fewer than 200 detected genes are likely empty droplets, and genes 
detected in fewer than 3 cells are likely noise.

In [ ]:
import matplotlib.pyplot as plt

sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=80, facecolor='white')

# remove cells with fewer than 200 detected genes (likely empty droplets)
# remove genes detected in fewer than 3 cells (likely noise)
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

print(adata)

In [ ]:
# some genes appear twice in the dataset due to the GEO submission format
# make all gene names unique to avoid downstream errors
adata.var_names_make_unique()
print("Variable names made unique")
print(adata)


In [ ]:
# identify mitochondrial genes — high MT% indicates dying or damaged cells
# human mitochondrial genes start with 'MT-'
adata.var['mt'] = adata.var_names.str.startswith('MT-')

# calculate QC metrics for each cell: total counts, genes detected, MT percentage
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

# visualize QC metrics split by condition to check data quality before filtering
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True, groupby='condition')

In [ ]:
print("Before filtering:", adata.n_obs, "cells")

# remove likely doublets (cells with too many genes or counts)
adata = adata[adata.obs.n_genes_by_counts < 6000, :]
# remove remaining low quality cells
adata = adata[adata.obs.n_genes_by_counts > 200, :]
adata = adata[adata.obs.total_counts < 40000, :]
# MT cutoff set to 25% rather than the standard 5% used for blood cells
# solid tumor tissue has genuinely higher mitochondrial activity
adata = adata[adata.obs.pct_counts_mt < 25, :]

print("After filtering:", adata.n_obs, "cells")
print("\nCondition counts after filtering:")
print(adata.obs['condition'].value_counts())

In [ ]:
# plot QC metrics after filtering to confirm cutoffs worked as expected
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True, groupby='condition')

In [ ]:
# save QC filtered object as checkpoint before normalization
adata.write('../data/breast_cancer_qc.h5ad')
print("QC complete and saved!")
print(f"Final dataset: {adata.n_obs} cells x {adata.n_vars} genes")

In [ ]:
## Normalization and Feature Selection

Normalizing counts to make cells comparable regardless of sequencing depth, 
then selecting highly variable genes for downstream analysis.

In [ ]:
# normalize each cell to 10,000 total counts so cells are comparable regardless of sequencing depth
sc.pp.normalize_total(adata, target_sum=1e4)

# log transform to compress the range of highly expressed genes
sc.pp.log1p(adata)

# select highly variable genes using batch_key to find genes consistently variable
# across all 21 patients rather than just within one sample
# this helps avoid batch-specific artifacts driving the analysis
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5, batch_key='sample_id')
sc.pl.highly_variable_genes(adata)

print(f"Highly variable genes: {adata.var.highly_variable.sum()}")

In [ ]:
## Dimensionality Reduction

Preprocessing data before PCA by removing technical variation, then compressing 
2,499 highly variable genes into 50 principal components.

In [ ]:
# save the normalized counts before subsetting — needed for differential expression later
adata.raw = adata

# subset to highly variable genes only
adata = adata[:, adata.var.highly_variable]

# regress out technical variation so it doesn't drive clustering
sc.pp.regress_out(adata, ['total_counts', 'pct_counts_mt'])

# scale each gene to unit variance, cap at 10 to limit effect of outliers
sc.pp.scale(adata, max_value=10)

# run PCA — compress 2,499 gene dimensions into 50 principal components
sc.tl.pca(adata, svd_solver='arpack')

# plot variance ratio to decide how many PCs to use downstream
sc.pl.pca_variance_ratio(adata, log=True)

print(adata)

In [ ]:
# save PCA object as checkpoint before neighbor graph and UMAP computation
adata.write('../data/breast_cancer_pca.h5ad')
print("Saved!")

In [ ]:
## Clustering and UMAP

Building a k-nearest neighbor graph using 30 PCs, then running UMAP for 
visualization and Leiden clustering to identify cell populations.

In [ ]:
# build k-nearest neighbor graph using top 30 PCs
# used 30 PCs rather than standard 10 because variance ratio plot showed gradual decline
# suggesting more complex biological variation than a typical blood cell dataset
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)

# compute UMAP embedding for visualization
sc.tl.umap(adata)

# plot UMAP colored by condition to see initial tumor vs normal separation
sc.pl.umap(adata, color='condition', title='Tumor vs Normal')

In [ ]:
# run Leiden clustering to identify cell populations
# resolution=0.5 controls granularity — higher = more clusters, lower = fewer
# 0.5 is a standard starting point for this type of analysis
sc.tl.leiden(adata, resolution=0.5)

# plot UMAP side by side: clusters and condition
# comparing both plots helps match which clusters belong to tumor vs normal
sc.pl.umap(adata, color=['leiden', 'condition'], ncols=2)

In [ ]:
## Cell Type Annotation

Finding marker genes for each cluster using the Wilcoxon rank-sum test, then 
annotating each cluster with a biological cell type identity.

In [ ]:
# find marker genes for each cluster using Wilcoxon rank-sum test
# compares each cluster against all other cells to find uniquely expressed genes
sc.tl.rank_genes_groups(adata, 'leiden', method='wilcoxon')

# plot top 5 marker genes per cluster
sc.pl.rank_genes_groups(adata, n_genes=5, sharey=False)

In [ ]:
# map cluster numbers to biological cell type labels
# annotations based on top marker genes from rank_genes_groups
# validated against known cell type signatures (Wu et al. 2021)
cell_type_map = {
    '0': 'Immune cells',
    '1': 'Luminal epithelial',
    '2': 'Basal epithelial',
    '3': 'Luminal epithelial 2',
    '4': 'Stressed cells',
    '5': 'Stromal cells',
    '6': 'Macrophages',
    '7': 'Monocytes',
    '8': 'Proliferating cells',
    '9': 'High MT cells',
    '10': 'Ribosomal high',
    '11': 'Endothelial/Stromal',
    '12': 'Fibroblasts',
    '13': 'Epithelial cells',
    '14': 'Plasma B cells',
    '15': 'Dendritic cells',
    '16': 'Mast cells',
    '17': 'CAFs',                # cancer-associated fibroblasts
    '18': 'NK/B cells',
    '19': 'Endothelial cells',
    '20': 'T/NK cells'
}

# apply cell type labels to each cell based on its cluster assignment
adata.obs['cell_type'] = adata.obs['leiden'].map(cell_type_map)

# plot final annotated UMAP
sc.pl.umap(adata, color='cell_type', legend_loc='on data', 
           title='Breast Cancer Cell Types', legend_fontsize=8)

In [ ]:
# save final annotated object — this is the file loaded by 02_analysis.ipynb
adata.write('../data/breast_cancer_annotated.h5ad')

# plot cell types alongside condition for final overview
sc.pl.umap(adata, color=['cell_type', 'condition'], 
           ncols=2, legend_fontsize=7,
           save='_breast_cancer_celltypes.png')

In [ ]:
## Cell Type Composition

Calculating the proportion of each cell type that comes from tumor vs normal 
tissue to identify tumor-specific and normal-enriched populations.

In [ ]:
import pandas as pd

# count cells per cell type per condition
proportions = adata.obs.groupby(['cell_type', 'condition'], observed=False).size().unstack(fill_value=0)

# calculate total cells and percentage from each condition per cell type
proportions['total'] = proportions.sum(axis=1)
proportions['pct_tumor'] = (proportions['Tumor_TN'] / proportions['total'] * 100).round(1)
proportions['pct_normal'] = (proportions['Normal'] / proportions['total'] * 100).round(1)

# sort by tumor percentage — most tumor-enriched cell types appear first
print(proportions[['Normal', 'Tumor_TN', 'pct_normal', 'pct_tumor']].sort_values('pct_tumor', ascending=False))

In [ ]:
import matplotlib.pyplot as plt

# stacked horizontal bar chart showing cell counts per condition for each cell type
# sorted by tumor count so most tumor-enriched cell types appear at the top
ax = proportions[['Normal', 'Tumor_TN']].sort_values('Tumor_TN').plot(
    kind='barh',        # horizontal bars make cell type labels easier to read
    stacked=True,       # stacked shows total cells and condition split simultaneously
    figsize=(10, 8),
    color=['#4C72B0', '#DD8452']  # blue = Normal, orange = Tumor_TN
)
plt.title('Cell Type Composition: Normal vs Tumor_TN', fontsize=14)
plt.xlabel('Number of cells')
plt.tight_layout()
plt.savefig('../figures/cell_type_proportions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved!")

In [ ]:
marker_genes = {
    'Luminal epithelial': ['KRT18', 'KRT7'],
    'Basal epithelial': ['KRT5', 'KRT17'],
    'Fibroblasts': ['DCN', 'LUM'],
    'CAFs': ['COL1A1', 'COL1A2'],
    'Macrophages': ['LYZ', 'CD74'],
    'Proliferating cells': ['HMGB2', 'STMN1'],
    'Stressed cells': ['DDIT4', 'HMOX1'],
    'Endothelial cells': ['VWF', 'PECAM1'],
    'Plasma B cells': ['MZB1', 'JCHAIN'],
    'T/NK cells': ['NKG7', 'GZMB']
}

sc.pl.dotplot(adata, marker_genes, groupby='cell_type', 
              dendrogram=False,
              title='Marker Gene Expression by Cell Type',
              save='_marker_genes.png')

## Summary
Full preprocessing pipeline complete. The annotated dataset is saved to `data/breast_cancer_annotated.h5ad` and is ready for analysis in  `02_analysis.ipynb`.

**Final dataset:** 117,232 cells × 2,499 highly variable genes  
**Conditions:** 57,870 Normal, 59,362 Tumor_TN  
**Cell populations identified:** 21